# Fine-Tuning LLaMA 3.1-8B-Instruct on Bengali Empathetic Conversations

This notebook fine-tunes LLaMA 3.1-8B-Instruct using LoRA/Unsloth for efficient training on Kaggle free GPUs.

## Requirements:
- Full-sequence tokenization (no sequence length reduction)
- LoRA or Unsloth for parameter-efficient fine-tuning
- Evaluation with Perplexity, BLEU, ROUGE
- Experiment logging


## 1. Setup and Installation


In [97]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

%pip install -q "transformers>=4.40.0,<5.0.0"
%pip install -q "peft>=0.8.0"
%pip install -q "datasets>=2.14.0"
%pip install -q "accelerate>=0.27.0"
%pip install -q "bitsandbytes>=0.41.0"
%pip install -q "sacrebleu>=2.3.0"
%pip install -q "rouge-score>=0.1.2"
%pip install -q "nltk>=3.8.0"
%pip install -q "trl>=0.7.0"



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [98]:
%pip install -q unsloth

Note: you may need to restart the kernel to use updated packages.


In [99]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

Transformers: 4.57.6
CUDA: True


In [100]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import torch
from torch.utils.data import Dataset, DataLoader

from unsloth import FastLanguageModel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset, Dataset as HFDataset
import nltk
from sacrebleu import BLEU
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n" + "=" * 60)
print("KAGGLE INPUT DIRECTORIES")
print("=" * 60)
kaggle_input = "/kaggle/input"
if os.path.exists(kaggle_input):
    print(f"\nContents of {kaggle_input}:")
    for item in os.listdir(kaggle_input):
        item_path = os.path.join(kaggle_input, item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                subdirs = os.listdir(item_path)
                for subdir in subdirs[:5]:
                    print(f"      └─ {subdir}")
                if len(subdirs) > 5:
                    print(f"      └─ ... and {len(subdirs) - 5} more")
            except:
                pass
        else:
            print(f"  📄 {item}")
else:
    print(f"Directory {kaggle_input} does not exist (not running in Kaggle)")


Using device: cuda
GPU: Tesla T4
GPU Memory: 15.64 GB

KAGGLE INPUT DIRECTORIES

Contents of /kaggle/input:
  📁 datasets/
      └─ raseluddin
  📁 models/
      └─ serhiikharchuk


## 2. Configuration


In [101]:
MODEL_PATH = "/kaggle/input/models/serhiikharchuk/llama-3.1-storm-8b/transformers/llama-3.1-storm-8b-bf16/1"
MODEL_NAME = MODEL_PATH
USE_UNSLOTH = True

LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "lora_dropout": 0.1,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM
}

TRAINING_CONFIG = {
    "output_dir": "./results",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "warmup_steps": 100,
    "logging_steps": 10,
    "eval_steps": 500,
    "save_steps": 500,
    "save_total_limit": 3,
    "fp16": False,
    "bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    "gradient_checkpointing": True,
    "optim": "adamw_torch",
    "lr_scheduler_type": "cosine",
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False
}

DATASET_CONFIG = {
    "max_length": 2048,
    "do_not_reduce_sequence": True,
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1
}

print("Configuration loaded successfully!")


Configuration loaded successfully!


In [102]:
class DatasetProcessor:
    """
    Processes Bengali Empathetic Conversations dataset for LLM fine-tuning.
    """
    
    def __init__(self, tokenizer, max_length: int = 2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def preprocess_text(self, text: str) -> str:
        """Preprocess Bengali text (normalization, cleaning)."""
        if text is None:
            return ""
        text = str(text)
        text = text.strip()
        return text
    
    def format_for_instruction(self, question: str, answer: str) -> str:
        """Format conversation for instruction tuning."""
        formatted = f"""### Instruction:
You are a helpful and empathetic Bengali assistant.

{question}

{answer}"""
        return formatted
    
    def tokenize_function(self, examples: Dict) -> Dict:
        """Tokenize examples with full sequence length preservation."""
        texts = []
        
        questions = examples.get('Questions', examples.get('input', []))
        answers = examples.get('Answers', examples.get('output', []))
        
        if not isinstance(questions, list):
            questions = [questions]
        if not isinstance(answers, list):
            answers = [answers]
        
        for i in range(len(questions)):
            question = questions[i] if i < len(questions) else ""
            answer = answers[i] if i < len(answers) else ""
            
            question = self.preprocess_text(question)
            answer = self.preprocess_text(answer)
            
            formatted_text = self.format_for_instruction(question, answer)
            texts.append(formatted_text)
        
        tokenized = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        
        tokenized["labels"] = tokenized["input_ids"].copy()
        
        return tokenized
    
    def process_dataset(self, dataset, split: str = "train") -> HFDataset:
        """Process and tokenize dataset."""
        print(f"Processing {split} dataset...")
        
        tokenized_dataset = dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=dataset.column_names,
            desc=f"Tokenizing {split}"
        )
        
        print(f"{split} dataset processed. Size: {len(tokenized_dataset)}")
        return tokenized_dataset

print("DatasetProcessor class defined!")


DatasetProcessor class defined!


In [103]:
print("Loading dataset from Kaggle input...")
full_dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/*.csv"
)

full_dataset = full_dataset["train"]

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"Number of rows: {len(full_dataset)}")
print(f"Column names: {full_dataset.column_names}")
print(f"Features: {full_dataset.features}")

print("\n" + "=" * 60)
print("SAMPLE DATA (First 3 rows)")
print("=" * 60)
for i in range(min(3, len(full_dataset))):
    example = full_dataset[i]
    print(f"\nExample {i+1}:")
    for key, value in example.items():
        print(f"  {key}: {str(value)[:100]}...")

print("\n" + "=" * 60)
print("CLEANING DATASET (removing None/empty values)")
print("=" * 60)

def filter_empty(example):
    """Filter out examples with None or empty Questions/Answers."""
    question = example.get('Questions', '')
    answer = example.get('Answers', '')
    return (
        question is not None and 
        answer is not None and 
        str(question).strip() != '' and 
        str(answer).strip() != ''
    )

original_size = len(full_dataset)
full_dataset = full_dataset.filter(filter_empty)
cleaned_size = len(full_dataset)
removed = original_size - cleaned_size

print(f"Original size: {original_size}")
print(f"After cleaning: {cleaned_size}")
print(f"Removed: {removed} rows ({100*removed/original_size:.2f}%)")

dataset = full_dataset.train_test_split(test_size=DATASET_CONFIG["test_split"] + DATASET_CONFIG["val_split"], seed=42)
val_test = dataset["test"].train_test_split(test_size=DATASET_CONFIG["test_split"] / (DATASET_CONFIG["test_split"] + DATASET_CONFIG["val_split"]), seed=42)

train_dataset = dataset["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"\nDataset splits:")
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")


Loading dataset from Kaggle input...
DATASET INFORMATION
Number of rows: 38233
Column names: ['Topics', 'Question-Title', 'Questions', 'Answers']
Features: {'Topics': Value('string'), 'Question-Title': Value('string'), 'Questions': Value('string'), 'Answers': Value('string')}

SAMPLE DATA (First 3 rows)

Example 1:
  Topics: পারিবারিক দ্বন্দ্ব...
  Question-Title: মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি...
  Questions:  আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চলছে। অতীতে, তাদের মধ্যে ছোটখাটো পার্থক্য ছিল। উদাহরণস্...
  Answers:  আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা "ত্রিভুজকরণ" বলে অভিহিত করেছেন। যা হয় যখন পরিবারের একজন সদ...

Example 2:
  Topics: পদার্থের অপব্যবহার, আসক্তি...
  Question-Title: আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?...
  Questions:  আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ধূমপান ছেড়ে দিতে হবে - কিন্তু এটা কঠিন। কখনও কখনও এটি...
  Answers:  হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্যকর তা করার জন্য আগে থেকে পরিকল্পনা করা আপনার জন্য ভাল...

Example 3:
  T

## 5. Initialize Tokenizer and Model


In [104]:
print("Loading tokenizer from Kaggle model...")
print(f"Model path: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer loaded successfully!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")
print("\nNote: If using Unsloth, tokenizer will be reloaded with optimizations in the next cell.")


Loading tokenizer from Kaggle model...
Model path: /kaggle/input/models/serhiikharchuk/llama-3.1-storm-8b/transformers/llama-3.1-storm-8b-bf16/1
Tokenizer loaded successfully!
Vocab size: 128000
Model max length: 131072
Pad token: <|end_of_text|>
EOS token: <|eot_id|>

Note: If using Unsloth, tokenizer will be reloaded with optimizations in the next cell.


In [105]:
processor = DatasetProcessor(tokenizer, max_length=DATASET_CONFIG["max_length"])
print("DatasetProcessor initialized!")


DatasetProcessor initialized!


In [106]:
if USE_UNSLOTH:
    print("Loading model with Unsloth from Kaggle input...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=DATASET_CONFIG["max_length"],
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    tokenizer.padding_side = "right"
    
    print(f"\nTokenizer configuration:")
    print(f"  Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
    print(f"  EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
    print(f"  Padding side: {tokenizer.padding_side}")
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_CONFIG["r"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias="none",
        use_gradient_checkpointing=True,
        random_state=3407
    )
    
    print("Model loaded with Unsloth and LoRA!")
    
else:
    print("Loading model with standard PEFT LoRA from Kaggle input...")
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        load_in_4bit=True,
        trust_remote_code=True
    )
    
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"]
    )
    
    model = get_peft_model(model, lora_config)
    
    print("Model loaded with PEFT LoRA!")

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params:,}")

print("\n" + "=" * 60)
print("MODEL INFORMATION")
print("=" * 60)
print(f"Model type: {type(model).__name__}")
if hasattr(model, 'config'):
    print(f"Model config: {model.config.name_or_path if hasattr(model.config, 'name_or_path') else 'N/A'}")
    print(f"Model dtype: {next(model.parameters()).dtype}")
    print(f"Device: {next(model.parameters()).device}")
    if hasattr(model.config, 'hidden_size'):
        print(f"Hidden size: {model.config.hidden_size}")
    if hasattr(model.config, 'num_hidden_layers'):
        print(f"Number of layers: {model.config.num_hidden_layers}")
    if hasattr(model.config, 'num_attention_heads'):
        print(f"Number of attention heads: {model.config.num_attention_heads}")
    if hasattr(model.config, 'max_position_embeddings'):
        print(f"Max position embeddings: {model.config.max_position_embeddings}")

processor = DatasetProcessor(tokenizer, max_length=DATASET_CONFIG["max_length"])
print("\nDatasetProcessor reinitialized with final tokenizer!")


Loading model with Unsloth from Kaggle input...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Tokenizer configuration:
  Pad token: <|end_of_text|> (ID: 128001)
  EOS token: <|eot_id|> (ID: 128009)
  Padding side: right
Model loaded with Unsloth and LoRA!

Trainable parameters: 13,631,488 (0.30%)
Total parameters: 4,554,231,808

MODEL INFORMATION
Model type: PeftModelForCausalLM
Model config: /kaggle/input/models/serhiikharchuk/llama-3.1-storm-8b/transformers/llama-3.1-storm-8b-bf16/1
Model dtype: torch.float16
Device: cuda:0
Hidden size: 4096
Number of layers: 32
Number of attention heads: 32
Max position embeddings: 131072

DatasetProcessor reinitialized with final tokenizer!


## 6. Process Datasets


In [107]:
print("=" * 60)
print("PROCESSING DATASETS")
print("=" * 60)
print("Note: Processing with final tokenizer (after Unsloth reload if applicable)")

train_tokenized = processor.process_dataset(train_dataset, split="train")
val_tokenized   = processor.process_dataset(val_dataset, split="val")
test_tokenized  = processor.process_dataset(test_dataset, split="test")

print("\n" + "=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)
print(f"Train tokenized: {len(train_tokenized)} samples")
print(f"Val tokenized: {len(val_tokenized)} samples")
print(f"Test tokenized: {len(test_tokenized)} samples")

print("\nDataset kept in list format (data collator will convert to tensors)")


PROCESSING DATASETS
Note: Processing with final tokenizer (after Unsloth reload if applicable)
Processing train dataset...


Tokenizing train:   0%|          | 0/30568 [00:00<?, ? examples/s]

train dataset processed. Size: 30568
Processing val dataset...


Tokenizing val:   0%|          | 0/3821 [00:00<?, ? examples/s]

val dataset processed. Size: 3821
Processing test dataset...


Tokenizing test:   0%|          | 0/3821 [00:00<?, ? examples/s]

test dataset processed. Size: 3821

PROCESSING COMPLETE
Train tokenized: 30568 samples
Val tokenized: 3821 samples
Test tokenized: 3821 samples

Dataset kept in list format (data collator will convert to tensors)


In [108]:
print("=" * 60)
print("INSPECTING PROCESSED DATA")
print("=" * 60)

print("\nExample from TRAIN dataset (first sample):")
sample = train_tokenized[0]
print(f"Keys: {sample.keys()}")
print(f"Input IDs length: {len(sample['input_ids'])}")
print(f"Labels length: {len(sample['labels'])}")

decoded_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f"\nDecoded text (first 500 chars):")
print(decoded_text[:500] + "...")

max_samples_for_stats = 1000
train_sample_size = min(max_samples_for_stats, len(train_tokenized))
val_sample_size = min(max_samples_for_stats, len(val_tokenized))
test_sample_size = min(max_samples_for_stats, len(test_tokenized))

train_lengths = [len(train_tokenized[i]['input_ids']) for i in range(train_sample_size)]
val_lengths = [len(val_tokenized[i]['input_ids']) for i in range(val_sample_size)]
test_lengths = [len(test_tokenized[i]['input_ids']) for i in range(test_sample_size)]

print("\n" + "=" * 60)
print("TOKEN LENGTH STATISTICS")
print("=" * 60)
print(f"\nTrain set (sampled {train_sample_size} of {len(train_tokenized)}):")
print(f"  Min length: {min(train_lengths)}")
print(f"  Max length: {max(train_lengths)}")
print(f"  Mean length: {np.mean(train_lengths):.1f}")
print(f"  Median length: {np.median(train_lengths):.1f}")
truncated_count = sum(1 for l in train_lengths if l > DATASET_CONFIG['max_length'])
print(f"  Samples > {DATASET_CONFIG['max_length']}: {truncated_count} ({100*truncated_count/train_sample_size:.1f}%)")

print(f"\nVal set (sampled {val_sample_size} of {len(val_tokenized)}):")
print(f"  Min length: {min(val_lengths)}")
print(f"  Max length: {max(val_lengths)}")
print(f"  Mean length: {np.mean(val_lengths):.1f}")
print(f"  Median length: {np.median(val_lengths):.1f}")

print(f"\nTest set (sampled {test_sample_size} of {len(test_tokenized)}):")
print(f"  Min length: {min(test_lengths)}")
print(f"  Max length: {max(test_lengths)}")
print(f"  Mean length: {np.mean(test_lengths):.1f}")
print(f"  Median length: {np.median(test_lengths):.1f}")

print("\n" + "=" * 60)
print("MORE EXAMPLES (showing formatted text)")
print("=" * 60)
for i in range(min(2, len(train_tokenized))):
    sample = train_tokenized[i]
    decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    print(f"\nExample {i+1} (length: {len(sample['input_ids'])} tokens):")
    print(decoded[:300] + "..." if len(decoded) > 300 else decoded)


INSPECTING PROCESSED DATA

Example from TRAIN dataset (first sample):
Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs length: 230
Labels length: 230

Decoded text (first 500 chars):
<|begin_of_text|>### Instruction:
You are a helpful and empathetic Bengali assistant.

### Input:
আমি এখন পর্যন্ত সেরা কাজ করেছি...সম্পূর্ণ স্ব-নিযুক্ত, ভাল খাওয়া, মহান বন্ধুদের সঙ্গ...মনে হচ্ছে জীবন আমাকে একটু শিথিল করতে শুরু করেছে

### Response:
এটি খুব সুন্দর, কারণ এটি সাধারণত ঘটে না।...

TOKEN LENGTH STATISTICS

Train set (sampled 1000 of 30568):
  Min length: 50
  Max length: 2048
  Mean length: 240.4
  Median length: 177.0
  Samples > 2048: 0 (0.0%)

Val set (sampled 1000 of 3821):
  Min length: 48
  Max length: 2048
  Mean length: 236.2
  Median length: 172.0

Test set (sampled 1000 of 3821):
  Min length: 48
  Max length: 2048
  Mean length: 231.3
  Median length: 173.0

MORE EXAMPLES (showing formatted text)

Example 1 (length: 230 tokens):
### Instruction:
You are a helpful a

## 7. Setup Training Arguments


In [109]:
import os
from transformers import TrainingArguments
from transformers.trainer_utils import IntervalStrategy

os.makedirs(TRAINING_CONFIG["output_dir"], exist_ok=True)

training_config = TRAINING_CONFIG.copy()

training_config["eval_strategy"] = IntervalStrategy.STEPS
training_config["save_strategy"] = IntervalStrategy.STEPS
training_config["eval_steps"] = 500
training_config["save_steps"] = 500
training_config["load_best_model_at_end"] = True

if "evaluation_strategy" in training_config:
    del training_config["evaluation_strategy"]

training_args = TrainingArguments(
    **training_config
)

print("Training arguments configured successfully!")
print(f"Evaluation strategy: {training_args.eval_strategy}")
print(f"Save strategy: {training_args.save_strategy}")
print(f"Load best model at end: {training_args.load_best_model_at_end}")




Training arguments configured successfully!
Evaluation strategy: IntervalStrategy.STEPS
Save strategy: SaveStrategy.STEPS
Load best model at end: True


## 8. Data Collator


In [116]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=None
)

print("Data collator created!")
print(f"Pad token ID: {tokenizer.pad_token_id}")
print(f"EOS token ID: {tokenizer.eos_token_id}")

print("\nVerifying tokenized data format...")
sample = train_tokenized[0]
print(f"Sample keys: {sample.keys()}")
print(f"Input IDs type: {type(sample['input_ids'])}, length: {len(sample['input_ids'])}")
print(f"Labels type: {type(sample['labels'])}, length: {len(sample['labels'])}")
print(f"Attention mask type: {type(sample.get('attention_mask', 'N/A'))}")

if isinstance(sample['labels'], list):
    print(f"✓ Labels are in list format (correct)")
    if len(sample['labels']) > 0:
        print(f"  First label value: {sample['labels'][0]} (type: {type(sample['labels'][0])})")
else:
    print(f"⚠ Labels are not in list format: {type(sample['labels'])}")



Data collator created!
Pad token ID: 128001
EOS token ID: 128009

Verifying tokenized data format...
Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs type: <class 'list'>, length: 230
Labels type: <class 'list'>, length: 230
Attention mask type: <class 'list'>
✓ Labels are in list format (correct)
  First label value: 128000 (type: <class 'int'>)


## 9. Experiment Logging


In [117]:
class ExperimentLogger:
    """Logs experiments to LLAMAExperiments table structure."""
    
    def __init__(self, log_dir: str = "./logs"):
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        self.experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiment_log = {
            "id": self.experiment_id,
            "model_path": MODEL_PATH,
            "lora_config": LORA_CONFIG,
            "training_config": TRAINING_CONFIG,
            "train_loss": [],
            "val_loss": [],
            "metrics": {},
            "timestamp": datetime.now().isoformat()
        }
        
    def log_training_step(self, step: int, loss: float):
        """Log training step loss."""
        self.experiment_log["train_loss"].append({"step": step, "loss": loss})
        
    def log_validation(self, step: int, val_loss: float, metrics: Dict = None):
        """Log validation results."""
        self.experiment_log["val_loss"].append({"step": step, "loss": val_loss})
        if metrics:
            self.experiment_log["metrics"][step] = metrics
        
    def save(self):
        """Save experiment log to JSON."""
        log_path = os.path.join(self.log_dir, f"experiment_{self.experiment_id}.json")
        with open(log_path, 'w', encoding='utf-8') as f:
            json.dump(self.experiment_log, f, indent=2, ensure_ascii=False)
        print(f"Experiment log saved to {log_path}")
        return log_path

logger = ExperimentLogger()
print(f"Experiment ID: {logger.experiment_id}")


Experiment ID: 20260219_045330


## 10. Custom Trainer with Logging


In [118]:
class CustomTrainer(Trainer):
    """Custom trainer with experiment logging."""
    
    def __init__(self, experiment_logger, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.experiment_logger = experiment_logger
    
    def log(self, logs: Dict[str, float], *args, **kwargs):
        """Override log to capture training metrics.
        
        Args:
            logs: Dictionary of metrics to log
            *args: Additional positional arguments (like start_time) that may be passed by Trainer
            **kwargs: Additional keyword arguments that may be passed by Trainer
        """
        super().log(logs, *args, **kwargs)
        
        if "loss" in logs:
            step = logs.get("step", 0)
            self.experiment_logger.log_training_step(step, logs["loss"])
        
        if "eval_loss" in logs:
            step = logs.get("step", 0)
            metrics = {k: v for k, v in logs.items() if k not in ["step", "eval_loss"]}
            self.experiment_logger.log_validation(step, logs["eval_loss"], metrics)

print("CustomTrainer class defined!")


CustomTrainer class defined!


## 11. Initialize Trainer


In [119]:
trainer = CustomTrainer(
    experiment_logger=logger,
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Trainer initialized!")


Trainer initialized!


## 12. Training


In [120]:
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Training on {len(train_tokenized)} samples")
print(f"Validating on {len(val_tokenized)} samples")
print(f"Total steps: {training_args.max_steps if hasattr(training_args, 'max_steps') and training_args.max_steps else 'N/A'}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size per device: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Evaluation every {training_args.eval_steps} steps")
print(f"Save every {training_args.save_steps} steps")
print("=" * 60)

try:
    train_result = trainer.train()
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print(f"Final training loss: {train_result.training_loss:.4f}")
    if hasattr(train_result, 'metrics'):
        print(f"\nTraining metrics:")
        for key, value in train_result.metrics.items():
            if isinstance(value, (int, float)):
                print(f"  {key}: {value:.4f}")
    
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
    print("Model state has been saved. You can resume training later.")
except Exception as e:
    print(f"\n❌ Training error: {str(e)}")
    print("Check the error message above for details.")
    raise


STARTING TRAINING
Training on 30568 samples
Validating on 3821 samples
Total steps: -1
Epochs: 3
Batch size per device: 1
Gradient accumulation steps: 8
Effective batch size: 8
Learning rate: 0.0002
Evaluation every 500 steps
Save every 500 steps


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 30,568 | Num Epochs = 3 | Total steps = 11,463
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss,Validation Loss



⚠️ Training interrupted by user
Model state has been saved. You can resume training later.


## 13. Save Model and Logs


In [ ]:
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)
    model.save_pretrained("lora_model")
else:
    trainer.save_model("lora_model")

tokenizer.save_pretrained("lora_model")
print("Model saved!")


In [ ]:
log_path = logger.save()
print(f"Experiment logged to: {log_path}")


## 14. Evaluation Metrics


In [ ]:
class Evaluator:
    """Evaluates model using Perplexity, BLEU, and ROUGE metrics."""
    
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.bleu = BLEU()
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    def calculate_perplexity(self, dataset, max_samples: int = 100):
        """Calculate perplexity on dataset."""
        self.model.eval()
        total_loss = 0
        total_tokens = 0
        
        eval_samples = min(max_samples, len(dataset))
        indices = np.random.choice(len(dataset), eval_samples, replace=False)
        
        with torch.no_grad():
            for idx in indices:
                sample = dataset[idx]
                input_ids = torch.tensor([sample['input_ids']]).to(device)
                labels = torch.tensor([sample['labels']]).to(device)
                
                outputs = self.model(input_ids=input_ids, labels=labels)
                loss = outputs.loss
                
                total_loss += loss.item() * input_ids.size(1)
                total_tokens += input_ids.size(1)
        
        avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
        perplexity = np.exp(avg_loss)
        
        self.model.train()
        return perplexity
    
    def calculate_bleu(self, predictions: List[str], references: List[str]):
        """Calculate BLEU score."""
        references_list = [[ref] for ref in references]
        bleu_score = self.bleu.corpus_score(predictions, references_list)
        return bleu_score.score / 100.0
    
    def calculate_rouge(self, predictions: List[str], references: List[str]):
        """Calculate ROUGE scores."""
        rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
        
        for pred, ref in zip(predictions, references):
            scores = self.rouge_scorer.score(ref, pred)
            rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
            rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
            rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)
        
        return {
            'rouge1': np.mean(rouge_scores['rouge1']),
            'rouge2': np.mean(rouge_scores['rouge2']),
            'rougeL': np.mean(rouge_scores['rougeL'])
        }
    
    def generate_response(self, question: str, max_length: int = 512) -> str:
        """Generate response given question."""
        self.model.eval()
        
        input_text = f"""### Instruction:
You are a helpful and empathetic Bengali assistant.

{question}

"""
        inputs = self.tokenizer(input_text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        if "### Response:" in generated_text:
            response = generated_text.split("### Response:")[-1].strip()
        else:
            response = generated_text[len(input_text):].strip()
        
        self.model.train()
        return response

print("Evaluator class defined!")


In [ ]:
evaluator = Evaluator(model, tokenizer)

print("Calculating perplexity on validation set...")
perplexity = evaluator.calculate_perplexity(val_tokenized, max_samples=50)
print(f"Perplexity: {perplexity:.4f}")

print("\nGenerating responses for BLEU/ROUGE evaluation...")
test_questions = [test_dataset[i]["Questions"] for i in range(min(10, len(test_dataset)))]
test_references = [test_dataset[i]["Answers"] for i in range(min(10, len(test_dataset)))]

predictions = []
for question in test_questions:
    response = evaluator.generate_response(question, max_length=256)
    predictions.append(response)
    print(f"\nQuestion: {question}")
    print(f"Generated: {response}")
    print(f"Reference: {test_references[len(predictions)-1]}")

bleu_score = evaluator.calculate_bleu(predictions, test_references)
rouge_scores = evaluator.calculate_rouge(predictions, test_references)

print(f"\n\n=== Evaluation Results ===")
print(f"Perplexity: {perplexity:.4f}")
print(f"BLEU Score: {bleu_score:.4f}")
print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")

final_metrics = {
    "perplexity": perplexity,
    "bleu": bleu_score,
    "rouge1": rouge_scores['rouge1'],
    "rouge2": rouge_scores['rouge2'],
    "rougeL": rouge_scores['rougeL']
}
logger.experiment_log["metrics"]["final"] = final_metrics
logger.save()


In [ ]:
generated_responses = []

for i, question in enumerate(test_questions):
    response = evaluator.generate_response(question, max_length=256)
    generated_responses.append({
        "id": i + 1,
        "experiment_id": logger.experiment_id,
        "input_text": question,
        "response_text": response,
        "timestamp": datetime.now().isoformat()
    })

responses_path = os.path.join("./logs", f"generated_responses_{logger.experiment_id}.json")
with open(responses_path, 'w', encoding='utf-8') as f:
    json.dump(generated_responses, f, indent=2, ensure_ascii=False)

print(f"Generated responses saved to {responses_path}")
print(f"Total responses: {len(generated_responses)}")


## 16. Summary and Next Steps


In [ ]:
print("=" * 50)
print("TRAINING SUMMARY")
print("=" * 50)
print(f"Experiment ID: {logger.experiment_id}")
print(f"Model: {MODEL_PATH}")
print(f"Method: {'Unsloth + LoRA' if USE_UNSLOTH else 'PEFT LoRA'}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Dataset: Bengali Empathetic Conversations Corpus")
print(f"Dataset size: {len(full_dataset)} rows")
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

if 'final_metrics' in globals():
    print(f"\nFinal Metrics:")
    for metric, value in final_metrics.items():
        print(f"  {metric}: {value:.4f}")
else:
    print("\nNote: Run evaluation cells to see final metrics")

if os.path.exists("lora_model"):
    print(f"\nModel saved to: lora_model/")
else:
    print(f"\nNote: Model not yet saved. Run save model cell.")

if 'log_path' in globals() and os.path.exists(log_path):
    print(f"Experiment log: {log_path}")
if 'responses_path' in globals() and os.path.exists(responses_path):
    print(f"Generated responses: {responses_path}")

print("=" * 50)
